In [14]:
import copy
# GLOBAL COUNTERS
backtrack_calls = 0
backtrack_failures = 0

# READ BOARD FROM FILE
def read_board(filename):
    board = []
    with open(filename, 'r') as f:
        for line in f:
            board.append([int(x) for x in line.strip()])
    return board

# PRINT BOARD
def print_board(board):
    for row in board:
        print(" ".join(str(x) for x in row))

# INITIALIZE DOMAINS
def get_domains(board):
    domains = {}
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                domains[(r, c)] = set(range(1, 10))
            else:
                domains[(r, c)] = {board[r][c]}
    return domains

# FIND NEIGHBORS
def neighbors(cell):
    r, c = cell
    nbrs = set()

    for i in range(9):
        nbrs.add((r, i))
        nbrs.add((i, c))

    br, bc = 3*(r//3), 3*(c//3)
    for i in range(br, br+3):
        for j in range(bc, bc+3):
            nbrs.add((i, j))

    nbrs.remove(cell)
    return nbrs

# AC-3 ALGORITHM
def ac3(domains):
    queue = [(xi, xj) for xi in domains for xj in neighbors(xi)]

    while queue:
        xi, xj = queue.pop(0)
        if revise(domains, xi, xj):
            if len(domains[xi]) == 0:
                return False
            for xk in neighbors(xi):
                if xk != xj:
                    queue.append((xk, xi))
    return True


# REVISE FUNCTION
def revise(domains, xi, xj):
    revised = False

    if len(domains[xj]) == 1:
        val = next(iter(domains[xj]))
        if val in domains[xi]:
            domains[xi].remove(val)
            revised = True

    return revised

# MRV HEURISTIC
def select_unassigned_variable(domains):
    return min(
        [v for v in domains if len(domains[v]) > 1],
        key=lambda x: len(domains[x])
    )

# BACKTRACKING SEARCH
def backtrack(domains):
    global backtrack_calls, backtrack_failures
    backtrack_calls += 1

    # If complete → return solution
    if all(len(domains[v]) == 1 for v in domains):
        return domains

    var = select_unassigned_variable(domains)

    for value in domains[var]:
        new_domains = copy.deepcopy(domains)
        new_domains[var] = {value}

        result = backtrack(new_domains)
        if result:
            return result

    backtrack_failures += 1
    return None

# SOLVE FUNCTION
def solve(filename, expected_calls, expected_fails):
    global backtrack_calls, backtrack_failures

    backtrack_calls = 0
    backtrack_failures = 0

    print("Solving:", filename)

    board = read_board(filename)
    domains = get_domains(board)

    if not ac3(domains):
        print("No solution found")
        return

    result = backtrack(domains)

    if result:
        solved = [[list(result[(r, c)])[0] for c in range(9)] for r in range(9)]
        print("\nSolution:")
        print_board(solved)
    else:
        print("No solution found")

    print("\nActual Stats:")
    print("Backtrack Calls:", backtrack_calls)
    print("Backtrack Failures:", backtrack_failures)

    print("\nExpected Stats (Assignment):")
    print("Backtrack Calls:", expected_calls)
    print("Backtrack Failures:", expected_fails)
# RUN ALL BOARDS
solve("easy.txt", 20, 2)
solve("medium.txt", 150, 40)
solve("hard.txt", 1200, 300)
solve("veryhard.txt", 5000, 1500)

Solving: easy.txt

Solution:
7 8 4 9 3 2 1 5 6
6 1 9 4 8 5 3 2 7
2 3 5 1 7 6 4 8 9
5 7 8 2 6 1 9 3 4
3 4 1 8 9 7 5 6 2
9 2 6 5 4 3 8 7 1
4 5 3 7 2 9 6 1 8
8 6 2 3 1 4 7 9 5
1 9 7 6 5 8 2 4 3

Actual Stats:
Backtrack Calls: 1
Backtrack Failures: 0

Expected Stats (Assignment):
Backtrack Calls: 20
Backtrack Failures: 2
Solving: medium.txt
No solution found
Solving: hard.txt
No solution found
Solving: veryhard.txt
No solution found
